In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from category_encoders import TargetEncoder
import warnings

warnings.filterwarnings('ignore')

print("\n--- 6. FORUM TAKTİĞİ: Target Encoding (Hedef Kodlama) ---")

# 1. Veriyi en baştan, tertemiz yüklüyoruz ki önceki One-Hot'lar karışmasın
train_te = pd.read_csv("../data/raw/train.csv", index_col='id')
test_te = pd.read_csv("../data/raw/test.csv", index_col='id')

y_te = train_te['Churn'].map({'Yes': 1, 'No': 0})
train_te.drop('Churn', axis=1, inplace=True)

# Kategorik kolonları otomatik seçiyoruz
cat_cols = train_te.select_dtypes(include=['object']).columns.tolist()

# Olası TotalCharges hatası düzeltmesi
train_te['TotalCharges'] = pd.to_numeric(train_te['TotalCharges'], errors='coerce').fillna(0)
test_te['TotalCharges'] = pd.to_numeric(test_te['TotalCharges'], errors='coerce').fillna(0)

print(f"Toplam {len(cat_cols)} adet kategorik değişkene Target Encoding uygulanacak...\n")

# 2. Cross-Validation ve Target Encoding (Sızıntıyı Önlemek İçin Aynı Döngüde)
skf_te = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds_te = np.zeros(len(train_te))
test_preds_te = np.zeros(len(test_te))

for fold, (train_idx, valid_idx) in enumerate(skf_te.split(train_te, y_te)):
    # Her fold için veriyi ayırıyoruz
    X_tr, y_tr = train_te.iloc[train_idx].copy(), y_te.iloc[train_idx]
    X_va, y_va = train_te.iloc[valid_idx].copy(), y_te.iloc[valid_idx]
    X_te_test = test_te.copy()
    
    # PROFESYONEL DOKUNUŞ: Sadece eğitim verisiyle Encoder'ı eğitiyoruz
    # smoothing=10 parametresi, az görülen kategorilerin skoru bozmasını engeller
    encoder = TargetEncoder(cols=cat_cols, smoothing=10)
    
    X_tr[cat_cols] = encoder.fit_transform(X_tr[cat_cols], y_tr)
    X_va[cat_cols] = encoder.transform(X_va[cat_cols])
    X_te_test[cat_cols] = encoder.transform(X_te_test[cat_cols])
    
    # 3. XGBoost Eğitimi
    model_te = XGBClassifier(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="auc"
    )
    model_te.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    
    valid_preds = model_te.predict_proba(X_va)[:, 1]
    oof_preds_te[valid_idx] = valid_preds
    
    # Test setini her fold'da tahmin edip topluyoruz
    test_preds_te += model_te.predict_proba(X_te_test)[:, 1] / skf_te.n_splits
    
    print(f"Target Encoding Fold {fold+1} AUC: {roc_auc_score(y_va, valid_preds):.4f}")

print(f"\n🚀 Target Encoding (OOF) AUC Skoru: {roc_auc_score(y_te, oof_preds_te):.4f}")

# 4. Gönderim Dosyası
submission_te = pd.DataFrame({'id': test_te.index, 'Churn': test_preds_te})
submission_te.to_csv("../submissions/submission_target_encoding.csv", index=False)
print("✅ Target Encoding gönderim dosyası 'submissions' klasörüne kaydedildi!")


--- 6. FORUM TAKTİĞİ: Target Encoding (Hedef Kodlama) ---
Toplam 15 adet kategorik değişkene Target Encoding uygulanacak...

Target Encoding Fold 1 AUC: 0.9157
Target Encoding Fold 2 AUC: 0.9168
Target Encoding Fold 3 AUC: 0.9161
Target Encoding Fold 4 AUC: 0.9172
Target Encoding Fold 5 AUC: 0.9145

🚀 Target Encoding (OOF) AUC Skoru: 0.9161
✅ Target Encoding gönderim dosyası 'submissions' klasörüne kaydedildi!
